# Gold Layer - Instacart Customer, Product and Revenue Analytics

This notebook creates business-ready analytical tables in the `big_data.gold` schema using clean and enriched data from the Silver layer.

## Business Problem

How can Instacart identify the most relevant products, departments, time periods, and customer segments to improve reorder strategies, product recommendations, and revenue analysis?

## Analytical Objectives

* Identify purchase patterns by period of day and hour.
* Analyze reorder behavior by product, aisle, and department.
* Segment customers according to purchase frequency and loyalty.
* Measure product and department performance using order volume, reorder rate, and estimated revenue.
* Support business decisions related to recommendation, retention, inventory, and sales strategy.

## Analysis Categories

* **executive_summary** - Global business KPIs including orders, customers, revenue, and reorder rate.
* **order_metrics** - Order-level metrics such as basket size, estimated revenue, and reorder rate.
* **sales_by_time_period** - Purchase and revenue metrics by period of day.
* **sales_by_hour** - Hourly purchase and revenue patterns.
* **customer_segmentation** - Customer loyalty segments based on order frequency.
* **customer_basket_by_segment** - Basket size and estimated order value by customer segment.
* **product_performance** - Product-level performance metrics.
* **department_performance** - Department-level performance metrics.
* **product_opportunity_ranking** - Products with high business potential based on volume, reorder rate, and revenue.
* **product_abc_analysis** - ABC classification of products based on cumulative estimated revenue.
* **slow_moving_products** - Products with low sales volume and low reorder behavior.
* **product_pairs** - Frequently purchased product combinations for cross-selling analysis.

In [0]:
# PySpark functions
from pyspark.sql import functions as F

from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    avg,
    sum as _sum,
    round as _round,
    max as _max,
    min as _min,
    when,
    percentile_approx,
    dense_rank,
    row_number,
    lit
)

from pyspark.sql.window import Window

In [0]:
# Schema configuration
silver_schema = "big_data.silver"
gold_schema = "big_data.gold"

# Silver tables used as analytical sources
silver_tables = [
    "orders",
    "products_enriched",
    "order_products"
]

# Create Gold schema if it does not exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

print(f"Source schema: {silver_schema}")
print(f"Target schema: {gold_schema}")
print(f"Silver tables loaded: {len(silver_tables)}")

In [0]:
# Display Silver table schemas and sample records
print("Silver Layer Validation\n")

for table_name in silver_tables:
    
    full_table_name = f"{silver_schema}.{table_name}"
    table_df = spark.table(full_table_name)

    print(f"\nTable: {full_table_name}")

    # Display schema
    print("Schema:")
    table_df.printSchema()

    # Display metadata
    print(f"Total rows: {table_df.count():,}")
    print(f"Total columns: {len(table_df.columns)}")

    # Display sample records
    print("Sample records:")
    display(table_df.limit(3))

## Sales Analysis by Time Period and Hour

In [0]:
## Order and Revenue Analysis by Time Period and Hour

orders = spark.table(f"{silver_schema}.orders")
order_products = spark.table(f"{silver_schema}.order_products")
products = spark.table(f"{silver_schema}.products_enriched")

orders_with_products = orders \
    .join(order_products, "order_id") \
    .join(products, "product_id")

total_orders = orders.select("order_id").distinct().count()

print("Order and Revenue Analysis by Time Period\n")

sales_by_period = orders_with_products.groupBy("period_of_day") \
    .agg(
        countDistinct("order_id").alias("total_orders"),
        count("product_id").alias("total_items_sold"),
        _round(_sum("price_usd"), 2).alias("estimated_revenue_usd"),
        _round(avg("price_usd"), 2).alias("avg_item_price_usd")
    ) \
    .withColumn(
        "percentage_orders",
        _round((col("total_orders") / total_orders) * 100, 2)
    ) \
    .withColumn(
        "avg_items_per_order",
        _round(col("total_items_sold") / col("total_orders"), 2)
    ) \
    .withColumn(
        "avg_order_value_usd",
        _round(col("estimated_revenue_usd") / col("total_orders"), 2)
    ) \
    .orderBy(col("total_orders").desc())

print("Orders and Revenue by Period of Day:")
display(sales_by_period)

sales_by_hour = orders_with_products.groupBy("order_hour_of_day") \
    .agg(
        countDistinct("order_id").alias("total_orders"),
        count("product_id").alias("total_items_sold"),
        _round(_sum("price_usd"), 2).alias("estimated_revenue_usd"),
        _round(avg("price_usd"), 2).alias("avg_item_price_usd")
    ) \
    .withColumn(
        "percentage_orders",
        _round((col("total_orders") / total_orders) * 100, 2)
    ) \
    .withColumn(
        "avg_items_per_order",
        _round(col("total_items_sold") / col("total_orders"), 2)
    ) \
    .withColumn(
        "avg_order_value_usd",
        _round(col("estimated_revenue_usd") / col("total_orders"), 2)
    ) \
    .orderBy("order_hour_of_day")

print("\nOrders and Revenue by Hour of Day:")
display(sales_by_hour)

## Reorder Analysis by Product Category (Department and Aisle)

In [0]:
## Reorder and Revenue Analysis by Product Category

order_products = spark.table(f"{silver_schema}.order_products")
products_enriched = spark.table(f"{silver_schema}.products_enriched")

print("Reorder and Revenue Analysis by Product Category\n")

# Join order items with product category and price information
orders_with_categories = order_products.join(products_enriched, "product_id")

# Reorder analysis by department
reorders_by_dept = orders_with_categories.groupBy("department") \
    .agg(
        count("*").alias("total_items"),
        countDistinct("product_id").alias("unique_products"),
        _sum(when(col("reordered") == True, 1).otherwise(0)).alias("reordered_items"),
        _round(_sum("price_usd"), 2).alias("estimated_revenue_usd"),
        _round(avg("price_usd"), 2).alias("avg_item_price_usd")
    ) \
    .withColumn(
        "reorder_rate",
        _round((col("reordered_items") / col("total_items")) * 100, 2)
    ) \
    .orderBy(col("reordered_items").desc())

print("Reorders by Department:")
display(reorders_by_dept)

# Reorder analysis by aisle
reorders_by_aisle = orders_with_categories.groupBy("aisle") \
    .agg(
        count("*").alias("total_items"),
        countDistinct("product_id").alias("unique_products"),
        _sum(when(col("reordered") == True, 1).otherwise(0)).alias("reordered_items"),
        _round(_sum("price_usd"), 2).alias("estimated_revenue_usd"),
        _round(avg("price_usd"), 2).alias("avg_item_price_usd")
    ) \
    .withColumn(
        "reorder_rate",
        _round((col("reordered_items") / col("total_items")) * 100, 2)
    ) \
    .orderBy(col("reordered_items").desc())

print("\nReorders by Aisle:")
display(reorders_by_aisle.limit(10))

## Analysis of Days Between Customer Orders

In [0]:
## Customer Purchase Frequency and Recency Analysis

orders = spark.table(f"{silver_schema}.orders")

print("Customer Purchase Frequency and Recency Analysis\n")

# Overall statistics for repeat orders
overall_purchase_frequency = orders.filter(col("is_first_order") == False) \
    .agg(
        count("*").alias("total_repeat_orders"),
        _round(avg("days_since_prior_order"), 2).alias("avg_days_between_orders"),
        _round(_min("days_since_prior_order"), 2).alias("min_days_between_orders"),
        _round(_max("days_since_prior_order"), 2).alias("max_days_between_orders"),
        _round(percentile_approx("days_since_prior_order", 0.5), 2).alias("median_days_between_orders")
    )

print("Overall Purchase Frequency Statistics:")
display(overall_purchase_frequency)

# Average days between orders by customer
customer_purchase_frequency = orders.filter(col("is_first_order") == False) \
    .groupBy("user_id") \
    .agg(
        count("*").alias("repeat_order_count"),
        _round(avg("days_since_prior_order"), 2).alias("avg_days_between_orders"),
        _round(_min("days_since_prior_order"), 2).alias("min_days_between_orders"),
        _round(_max("days_since_prior_order"), 2).alias("max_days_between_orders")
    ) \
    .withColumn(
        "purchase_frequency_segment",
        when(col("avg_days_between_orders") <= 7, "Weekly")
        .when(col("avg_days_between_orders") <= 14, "Biweekly")
        .when(col("avg_days_between_orders") <= 30, "Monthly")
        .otherwise("Occasional")
    ) \
    .orderBy(col("repeat_order_count").desc())

print("\nCustomer Purchase Frequency:")
display(customer_purchase_frequency.limit(10))

# Distribution of days between orders
days_distribution = orders.filter(col("is_first_order") == False) \
    .withColumn(
        "days_range",
        when(col("days_since_prior_order") <= 7, "1-7 days")
        .when(col("days_since_prior_order") <= 14, "8-14 days")
        .when(col("days_since_prior_order") <= 21, "15-21 days")
        .when(col("days_since_prior_order") <= 30, "22-30 days")
        .otherwise("30+ days")
    ) \
    .groupBy("days_range") \
    .agg(count("*").alias("order_count")) \
    .withColumn(
        "range_order",
        when(col("days_range") == "1-7 days", 1)
        .when(col("days_range") == "8-14 days", 2)
        .when(col("days_range") == "15-21 days", 3)
        .when(col("days_range") == "22-30 days", 4)
        .otherwise(5)
    ) \
    .orderBy("range_order") \
    .drop("range_order")

print("\nDistribution of Days Between Orders:")
display(days_distribution)

## Customer Behavior Analysis and Segmentation Based on Orders

In [0]:
## Customer Behavior Analysis and Segmentation

# Load Silver tables
orders = spark.table(f"{silver_schema}.orders")
order_products = spark.table(f"{silver_schema}.order_products")
products = spark.table(f"{silver_schema}.products_enriched")

print("Customer Behavior Analysis and Segmentation\n")

# Customer order frequency
customer_orders = orders.groupBy("user_id") \
    .agg(
        countDistinct("order_id").alias("total_orders"),
        _max("order_number").alias("max_order_number"),
        _round(avg("days_since_prior_order"), 2).alias("avg_days_between_orders")
    )

# Customer segmentation by order frequency
customer_segments = customer_orders.withColumn(
    "customer_segment",
    when(col("total_orders") == 1, "New")
    .when(col("total_orders").between(2, 5), "Occasional")
    .when(col("total_orders").between(6, 10), "Regular")
    .when(col("total_orders").between(11, 20), "Frequent")
    .otherwise("Loyal")
)

# Distribution of customers by segment
customer_segment_distribution = customer_segments.groupBy("customer_segment") \
    .agg(
        count("*").alias("customer_count"),
        _round(avg("total_orders"), 2).alias("avg_orders_per_customer")
    ) \
    .withColumn(
        "segment_order",
        when(col("customer_segment") == "New", 1)
        .when(col("customer_segment") == "Occasional", 2)
        .when(col("customer_segment") == "Regular", 3)
        .when(col("customer_segment") == "Frequent", 4)
        .otherwise(5)
    ) \
    .orderBy("segment_order") \
    .drop("segment_order")

print("Customer Segment Distribution:")
display(customer_segment_distribution)

# Order-level basket metrics
customer_basket = orders \
    .join(order_products, "order_id") \
    .join(products, "product_id") \
    .groupBy("user_id", "order_id") \
    .agg(
        count("product_id").alias("items_in_basket"),
        _round(_sum("price_usd"), 2).alias("estimated_order_value_usd"),
        _sum(when(col("reordered") == True, 1).otherwise(0)).alias("reordered_items")
    ) \
    .withColumn(
        "order_reorder_rate",
        _round((col("reordered_items") / col("items_in_basket")) * 100, 2)
    )

# Average basket and revenue by customer segment
avg_basket_by_segment = customer_segments.join(customer_basket, "user_id") \
    .groupBy("customer_segment") \
    .agg(
        countDistinct("user_id").alias("total_customers"),
        countDistinct("order_id").alias("total_orders"),
        _round(avg("items_in_basket"), 2).alias("avg_basket_size"),
        _round(avg("estimated_order_value_usd"), 2).alias("avg_order_value_usd"),
        _round(avg("order_reorder_rate"), 2).alias("avg_order_reorder_rate")
    ) \
    .withColumn(
        "segment_order",
        when(col("customer_segment") == "New", 1)
        .when(col("customer_segment") == "Occasional", 2)
        .when(col("customer_segment") == "Regular", 3)
        .when(col("customer_segment") == "Frequent", 4)
        .otherwise(5)
    ) \
    .orderBy("segment_order") \
    .drop("segment_order")

print("\nAverage Basket and Revenue by Customer Segment:")
display(avg_basket_by_segment)

# Top loyal customers
top_customers = customer_segments.join(customer_basket, "user_id") \
    .groupBy("user_id", "customer_segment") \
    .agg(
        countDistinct("order_id").alias("total_orders"),
        _sum("items_in_basket").alias("total_items_purchased"),
        _round(_sum("estimated_order_value_usd"), 2).alias("estimated_total_spent_usd"),
        _round(avg("items_in_basket"), 2).alias("avg_basket_size"),
        _round(avg("estimated_order_value_usd"), 2).alias("avg_order_value_usd"),
        _round(avg("order_reorder_rate"), 2).alias("avg_reorder_rate")
    ) \
    .orderBy(col("total_orders").desc(), col("estimated_total_spent_usd").desc())

print("\nTop 10 Customers by Order Frequency and Estimated Revenue:")
display(top_customers.limit(10))

# Customer lifecycle metrics
total_orders_count = orders.select("order_id").distinct().count()
total_customers_count = orders.select("user_id").distinct().count()

lifecycle_metrics = orders.agg(
    countDistinct("order_id").alias("total_orders"),
    countDistinct("user_id").alias("total_customers"),
    _sum(when(col("is_first_order") == True, 1).otherwise(0)).alias("first_time_orders"),
    _sum(when(col("is_first_order") == False, 1).otherwise(0)).alias("repeat_orders")
) \
.withColumn(
    "avg_orders_per_customer",
    _round(col("total_orders") / col("total_customers"), 2)
) \
.withColumn(
    "repeat_order_rate",
    _round((col("repeat_orders") / col("total_orders")) * 100, 2)
)

print("\nCustomer Lifecycle Metrics:")
display(lifecycle_metrics)

## Product and Category Performance Analysis in Sales

In [0]:
## Product and Category Performance Analysis

order_products = spark.table(f"{silver_schema}.order_products")
products_enriched = spark.table(f"{silver_schema}.products_enriched")
orders = spark.table(f"{silver_schema}.orders")

print("Product and Category Performance Analysis\n")

# Join order items with product, category, and price information
orders_with_products = order_products.join(products_enriched, "product_id")

# Product-level performance
product_performance = orders_with_products.groupBy(
    "product_id",
    "product_name",
    "department",
    "aisle",
    "price_band"
) \
.agg(
    count("*").alias("times_ordered"),
    countDistinct("order_id").alias("unique_orders"),
    _sum(when(col("reordered") == True, 1).otherwise(0)).alias("times_reordered"),
    _round(avg("price_usd"), 2).alias("avg_price_usd"),
    _round(_sum("price_usd"), 2).alias("estimated_revenue_usd")
) \
.withColumn(
    "reorder_rate",
    _round((col("times_reordered") / col("times_ordered")) * 100, 2)
) \
.orderBy(col("times_ordered").desc())

print("Top 10 Products by Order Volume:")
display(product_performance.limit(10))

# Top products by reorder rate with minimum order threshold
top_reorder_products = product_performance \
    .filter(col("times_ordered") >= 100) \
    .orderBy(col("reorder_rate").desc(), col("times_ordered").desc())

print("\nTop 10 Products by Reorder Rate (min 100 orders):")
display(top_reorder_products.limit(10))

# Department performance summary
department_performance = orders_with_products.groupBy("department") \
    .agg(
        countDistinct("product_id").alias("unique_products"),
        count("*").alias("total_items_sold"),
        countDistinct("order_id").alias("unique_orders"),
        _sum(when(col("reordered") == True, 1).otherwise(0)).alias("reordered_items"),
        _round(_sum("price_usd"), 2).alias("estimated_revenue_usd"),
        _round(avg("price_usd"), 2).alias("avg_item_price_usd")
    ) \
    .withColumn(
        "reorder_rate",
        _round((col("reordered_items") / col("total_items_sold")) * 100, 2)
    ) \
    .withColumn(
        "avg_items_per_order",
        _round(col("total_items_sold") / col("unique_orders"), 2)
    ) \
    .orderBy(col("estimated_revenue_usd").desc())

print("\nDepartment Performance Summary:")
display(department_performance)

# Add-to-cart order analysis
add_to_cart_analysis = orders_with_products.groupBy("add_to_cart_order") \
    .agg(
        count("*").alias("product_count"),
        _sum(when(col("reordered") == True, 1).otherwise(0)).alias("reordered_count"),
        _round(avg("price_usd"), 2).alias("avg_item_price_usd"),
        _round(_sum("price_usd"), 2).alias("estimated_revenue_usd")
    ) \
    .withColumn(
        "reorder_rate",
        _round((col("reordered_count") / col("product_count")) * 100, 2)
    ) \
    .filter(col("add_to_cart_order") <= 10) \
    .orderBy("add_to_cart_order")

print("\nProduct Add-to-Cart Order Analysis:")
display(add_to_cart_analysis)

# Cross-category purchase patterns
order_dept_diversity = orders_with_products.groupBy("order_id") \
    .agg(
        countDistinct("department").alias("department_count"),
        countDistinct("aisle").alias("aisle_count"),
        count("product_id").alias("basket_size"),
        _round(_sum("price_usd"), 2).alias("estimated_order_value_usd")
    )

diversity_distribution = order_dept_diversity.groupBy("department_count") \
    .agg(
        count("*").alias("order_count"),
        _round(avg("basket_size"), 2).alias("avg_basket_size"),
        _round(avg("estimated_order_value_usd"), 2).alias("avg_order_value_usd")
    ) \
    .orderBy("department_count")

print("\nCross-Category Purchase Patterns:")
display(diversity_distribution)

## Product Opportunity Ranking

In [0]:
print("Creating Product Opportunity Ranking\n")

# Products with strong reorder and revenue performance
product_opportunity_ranking = product_performance \
    .filter(col("times_ordered") >= 100) \
    .withColumn(
        "opportunity_score",
        _round(
            (
                col("reorder_rate") * 0.4 +
                col("times_ordered") * 0.0001 +
                col("estimated_revenue_usd") * 0.00001
            ),
            4
        )
    )

# Ranking window
ranking_window = Window.orderBy(col("opportunity_score").desc())

# Create ranking
product_opportunity_ranking = product_opportunity_ranking \
    .withColumn(
        "opportunity_rank",
        dense_rank().over(ranking_window)
    ) \
    .orderBy("opportunity_rank")

print("Top Product Opportunities:")
display(product_opportunity_ranking.limit(10))

## Order Metrics

In [0]:
print("Creating Order Metrics\n")

order_metrics = orders \
    .join(order_products, "order_id") \
    .join(products_enriched, "product_id") \
    .groupBy(
        "order_id",
        "user_id",
        "order_number",
        "order_dow",
        "order_hour_of_day",
        "period_of_day",
        "days_since_prior_order",
        "is_first_order"
    ) \
    .agg(
        count("product_id").alias("basket_size"),
        countDistinct("department").alias("unique_departments"),
        countDistinct("aisle").alias("unique_aisles"),
        _sum(when(col("reordered") == True, 1).otherwise(0)).alias("reordered_items"),
        _round(_sum("price_usd"), 2).alias("estimated_order_value_usd")
    ) \
    .withColumn(
        "order_reorder_rate",
        _round((col("reordered_items") / col("basket_size")) * 100, 2)
    )

display(order_metrics.limit(10))

## Aisle Performance

In [0]:
print("Creating Aisle Performance\n")

aisle_performance = orders_with_products.groupBy("department", "aisle") \
    .agg(
        countDistinct("product_id").alias("unique_products"),
        count("*").alias("total_items_sold"),
        countDistinct("order_id").alias("unique_orders"),
        _sum(when(col("reordered") == True, 1).otherwise(0)).alias("reordered_items"),
        _round(_sum("price_usd"), 2).alias("estimated_revenue_usd"),
        _round(avg("price_usd"), 2).alias("avg_item_price_usd")
    ) \
    .withColumn(
        "reorder_rate",
        _round((col("reordered_items") / col("total_items_sold")) * 100, 2)
    ) \
    .withColumn(
        "avg_items_per_order",
        _round(col("total_items_sold") / col("unique_orders"), 2)
    ) \
    .orderBy(col("estimated_revenue_usd").desc())

display(aisle_performance.limit(10))

## Customer Value Ranking

In [0]:
print("Creating Customer Value Ranking\n")

customer_value_ranking = customer_segments.join(customer_basket, "user_id") \
    .groupBy("user_id", "customer_segment") \
    .agg(
        countDistinct("order_id").alias("total_orders"),
        _sum("items_in_basket").alias("total_items_purchased"),
        _round(_sum("estimated_order_value_usd"), 2).alias("estimated_total_spent_usd"),
        _round(avg("items_in_basket"), 2).alias("avg_basket_size"),
        _round(avg("estimated_order_value_usd"), 2).alias("avg_order_value_usd"),
        _round(avg("order_reorder_rate"), 2).alias("avg_reorder_rate")
    )

customer_value_window = Window.orderBy(col("estimated_total_spent_usd").desc())

customer_value_ranking = customer_value_ranking \
    .withColumn("customer_value_rank", dense_rank().over(customer_value_window)) \
    .orderBy("customer_value_rank")

display(customer_value_ranking.limit(10))

## Price Band Performance

In [0]:
print("Creating Price Band Performance\n")

price_band_performance = orders_with_products.groupBy("price_band") \
    .agg(
        countDistinct("product_id").alias("unique_products"),
        count("*").alias("total_items_sold"),
        countDistinct("order_id").alias("unique_orders"),
        _sum(when(col("reordered") == True, 1).otherwise(0)).alias("reordered_items"),
        _round(_sum("price_usd"), 2).alias("estimated_revenue_usd"),
        _round(avg("price_usd"), 2).alias("avg_item_price_usd")
    ) \
    .withColumn(
        "reorder_rate",
        _round((col("reordered_items") / col("total_items_sold")) * 100, 2)
    ) \
    .withColumn(
        "avg_items_per_order",
        _round(col("total_items_sold") / col("unique_orders"), 2)
    ) \
    .orderBy(col("estimated_revenue_usd").desc())

display(price_band_performance)

## Product Pair Analysis

In [0]:
print("Creating Product Pair Analysis\n")

product_pairs = order_products.alias("a") \
    .join(
        order_products.alias("b"),
        (
            (col("a.order_id") == col("b.order_id")) &
            (col("a.product_id") < col("b.product_id"))
        )
    ) \
    .select(
        col("a.order_id").alias("order_id"),
        col("a.product_id").alias("product_a"),
        col("b.product_id").alias("product_b")
    )

product_pairs = product_pairs \
    .join(
        products_enriched.select(
            col("product_id").alias("product_a"),
            col("product_name").alias("product_a_name")
        ),
        "product_a"
    ) \
    .join(
        products_enriched.select(
            col("product_id").alias("product_b"),
            col("product_name").alias("product_b_name")
        ),
        "product_b"
    ) \
    .groupBy(
        "product_a",
        "product_a_name",
        "product_b",
        "product_b_name"
    ) \
    .agg(
        count("*").alias("pair_count")
    ) \
    .orderBy(col("pair_count").desc())

display(product_pairs.limit(10))

## Executive Business Summary

In [0]:
print("Creating Executive Business Summary\n")

# Order-level metrics
order_level_metrics = orders \
    .join(order_products, "order_id") \
    .join(products_enriched, "product_id") \
    .groupBy("order_id") \
    .agg(
        count("product_id").alias("basket_size"),
        _round(_sum("price_usd"), 2).alias("estimated_order_value_usd"),
        _sum(when(col("reordered") == True, 1).otherwise(0)).alias("reordered_items")
    )

# Executive summary table
executive_summary = order_level_metrics.agg(
    countDistinct("order_id").alias("total_orders"),
    _round(avg("basket_size"), 2).alias("avg_basket_size"),
    _round(avg("estimated_order_value_usd"), 2).alias("avg_order_value_usd"),
    _round(_sum("estimated_order_value_usd"), 2).alias("estimated_total_revenue_usd"),
    _sum("reordered_items").alias("total_reordered_items")
)

# Add global metrics
total_customers = orders.select("user_id").distinct().count()
total_products = products_enriched.select("product_id").distinct().count()
total_items = order_products.count()

executive_summary = executive_summary \
    .withColumn("total_customers", lit(total_customers)) \
    .withColumn("total_products", lit(total_products)) \
    .withColumn("total_items_sold", lit(total_items)) \
    .withColumn(
        "overall_reorder_rate",
        _round(
            (col("total_reordered_items") / col("total_items_sold")) * 100,
            2
        )
    )

display(executive_summary)

## Product ABC Analysis

In [0]:
print("Creating Product ABC Analysis\n")

# Product revenue aggregation
product_abc_analysis = product_performance \
    .select(
        "product_id",
        "product_name",
        "department",
        "times_ordered",
        "estimated_revenue_usd",
        "reorder_rate"
    ) \
    .orderBy(col("estimated_revenue_usd").desc())

# Window for cumulative revenue
abc_window = Window.orderBy(col("estimated_revenue_usd").desc())

# Total revenue
total_revenue = product_abc_analysis.agg(
    _sum("estimated_revenue_usd").alias("total_revenue")
).collect()[0]["total_revenue"]

# Cumulative revenue
product_abc_analysis = product_abc_analysis \
    .withColumn(
        "cumulative_revenue",
        _sum("estimated_revenue_usd").over(abc_window)
    ) \
    .withColumn(
        "cumulative_revenue_pct",
        _round(
            (col("cumulative_revenue") / lit(total_revenue)) * 100,
            2
        )
    )

# ABC classification
product_abc_analysis = product_abc_analysis \
    .withColumn(
        "abc_class",
        when(col("cumulative_revenue_pct") <= 80, "A")
        .when(col("cumulative_revenue_pct") <= 95, "B")
        .otherwise("C")
    )

display(product_abc_analysis.limit(10))

## Slow Moving Products

In [0]:
print("Creating Slow Moving Products\n")

slow_moving_products = product_performance \
    .filter(
        (col("times_ordered") < 50) &
        (col("reorder_rate") < 20)
    ) \
    .select(
        "product_id",
        "product_name",
        "department",
        "times_ordered",
        "reorder_rate",
        "estimated_revenue_usd",
        "avg_price_usd"
    ) \
    .orderBy(
        col("times_ordered").asc(),
        col("estimated_revenue_usd").asc()
    )

display(slow_moving_products.limit(10))

## Create Gold Layer Tables with Order, Revenue, Customer and Product Metrics

In [0]:
print("Creating Gold Layer Tables\n")

# Create gold schema if it does not exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
print(f"Schema {gold_schema} ready\n")

# Save Gold analytical tables
gold_tables = {
    "executive_summary": executive_summary,
    "order_metrics": order_metrics,
    "product_opportunity_ranking": product_opportunity_ranking,
    "sales_by_time_period": sales_by_period,
    "sales_by_hour": sales_by_hour,
    "reorder_analysis_by_department": reorders_by_dept,
    "reorder_analysis_by_aisle": reorders_by_aisle,
    "customer_purchase_frequency": days_distribution,
    "customer_segmentation": customer_segments,
    "customer_segment_distribution": customer_segment_distribution,
    "customer_basket_by_segment": avg_basket_by_segment,
    "customer_lifecycle_metrics": lifecycle_metrics,
    "product_performance": product_performance,
    "top_reorder_products": top_reorder_products,
    "department_performance": department_performance,
    "aisle_performance": aisle_performance,
    "customer_value_ranking": customer_value_ranking,
    "add_to_cart_analysis": add_to_cart_analysis,
    "price_band_performance": price_band_performance,
    "product_pairs": product_pairs,
    "cross_category_purchase_patterns": diversity_distribution,
    "product_abc_analysis": product_abc_analysis,
    "slow_moving_products": slow_moving_products,
}

for table_name, df in gold_tables.items():
    print(f"Creating {table_name}...")
    
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{gold_schema}.{table_name}")
    
    print(f"Created {gold_schema}.{table_name}")

print(f"\nAll Gold layer tables created successfully in {gold_schema}!")

print("\nGold Layer Validation\n")

# Show all Gold tables
spark.sql(f"SHOW TABLES IN {gold_schema}").show(truncate=False)

# Display row counts
print("\nGold Table Row Counts:\n")

for table_name in gold_tables.keys():
    row_count = spark.table(f"{gold_schema}.{table_name}").count()
    print(f"{table_name}: {row_count:,} rows")

In [0]:
print("Gold Layer Tables Created:\n")
spark.sql(f"SHOW TABLES IN {gold_schema}").show(truncate=False)

print("Gold Table Row Counts:\n")
for table_name in gold_tables.keys():
    row_count = spark.table(f"{gold_schema}.{table_name}").count()
    print(f"{table_name}: {row_count:,} rows")

## Business Problem Answer

The Gold layer was designed to answer the following business question:

**How can Instacart identify the most relevant products, departments, time periods, and customer segments to improve reorder strategies, product recommendations, revenue analysis, and inventory prioritization?**

Based on the analytical tables created in the Gold layer, the business problem can be answered through six main perspectives:

### Purchase timing and revenue behavior

The `sales_by_time_period`, `sales_by_hour`, and `order_metrics` tables identify the periods and hours with the highest order volume, item volume, estimated revenue, average basket size, and average order value.

### Reorder behavior

The `reorder_analysis_by_department`, `reorder_analysis_by_aisle`, `product_performance`, and `top_reorder_products` tables identify which categories and products have stronger reorder behavior and customer recurrence.

### Customer segmentation and loyalty

The `customer_segmentation`, `customer_segment_distribution`, `customer_basket_by_segment`, `customer_lifecycle_metrics`, and `customer_value_ranking` tables classify customers according to purchase frequency, loyalty level, estimated spending behavior, and reorder patterns.

### Product and category performance

The `product_performance`, `department_performance`, `aisle_performance`, `price_band_performance`, and `add_to_cart_analysis` tables show which products and categories generate higher sales volume, estimated revenue, customer recurrence, and stronger purchasing behavior.

### Strategic product analysis

The `product_opportunity_ranking`, `product_abc_analysis`, and `slow_moving_products` tables identify high-value products, revenue-driving products, and low-rotation products that may require inventory optimization or promotional strategies.

### Cross-selling and basket analysis

The `product_pairs` and `cross_category_purchase_patterns` tables identify products frequently purchased together and analyze category diversity within customer baskets, supporting recommendation systems and cross-selling opportunities.

Therefore, the Gold layer supports business decisions related to:

* Product recommendation
* Customer retention
* Inventory prioritization
* Revenue optimization
* Promotional campaigns
* Reorder prediction
* Cross-selling opportunities
* Product portfolio analysis

The analytical pipeline follows the Medallion Architecture approach, where:

* Raw layer stores original source files.
* Bronze layer stores raw structured Delta tables.
* Silver layer stores cleaned, typed, and enriched analytical datasets.
* Gold layer stores business-oriented aggregated metrics and strategic analytical tables.

In [0]:
print("\nGold Layer Validation\n")

spark.sql(f"SHOW TABLES IN {gold_schema}").show(truncate=False)

print("\nGold Table Row Counts:\n")

for table_name in gold_tables.keys():
    row_count = spark.table(f"{gold_schema}.{table_name}").count()
    print(f"{table_name}: {row_count:,} rows")

## Final Business Insights

The Gold layer provides business-ready analytical tables focused on customer behavior, reorder analysis, product performance, estimated revenue, and market basket patterns.

The generated datasets support strategic decision-making in areas such as:

* Product recommendation
* Inventory prioritization
* Customer retention
* Revenue optimization
* Purchase behavior analysis
* Reorder prediction
* Cross-selling opportunities

The architecture follows the Medallion Architecture approach, where:

* Raw layer stores original files.
* Bronze layer stores raw structured Delta tables.
* Silver layer stores cleaned and enriched analytical datasets.
* Gold layer stores business-oriented aggregated metrics and analytical tables.